# Banking Intent Classifier

Predicts which of 77 banking intents a customer message belongs to, using the BANKING77 dataset.

## Contents

1. Problem definition
2. Setup
3. Load the data
4. Explore the data
5. Preprocess and split the data

## 1. Problem definition

### 1.1 The problem

Bank customers often contact support with short, free-text messages. Before a chatbot can help, it has to work out what the customer wants: its **intent**.

This project builds a classifier that takes one customer message and predicts one of 77 banking intents.

| Example customer message | Intent |
|---|---|
| "My card hasn't arrived yet" | `card_arrival` |
| "I think someone has been using my card" | `compromised_card` |
| "Why was I charged twice for the same purchase?" | `transaction_charged_twice` |

This is **single-label, multi-class text classification**.

### 1.2 Users and their needs

| User | Need |
|---|---|
| **Bank customers** | Reach the right help quickly, without repeating themselves or being sent to the wrong place |
| **Support teams** | Have routine requests handled automatically, and receive the cases that need a person |
| **Chatbot developers** | A model that is accurate, fast, and clear about when it is unsure |

### 1.3 Constraints and challenges

- **Many classes:** 77 intents is far harder than a typical sentiment task with two or three labels.
- **Very similar intents:** several intents share much of their vocabulary and differ only in meaning, as shown below.
- **Short messages:** a message gives little context, and the chatbot has no conversation history to draw on.
- **Limited data:** about 13,000 labelled messages across 77 intents, so each intent has relatively few examples.
- **Informal text:** real customers make typos and use casual wording.
- **Off-topic messages:** a real chatbot receives messages outside its 77 intents, but BANKING77 has no examples of these.
- **Response time:** a chatbot has to reply quickly, so model size and speed matter as well as accuracy.

| Similar intents | The difference |
|---|---|
| `lost_or_stolen_card` / `compromised_card` | The customer no longer has the card, or the card's details have been misused |
| `card_arrival` / `card_delivery_estimate` | An expected card is missing, or the customer wants to know how long delivery takes |
| `pending_transfer` / `transfer_not_received_by_recipient` | The transfer has not completed, or it appears sent but the recipient does not have it |

### 1.4 The cost of errors

Not all mistakes are equally serious, so overall accuracy alone is not enough.

| Severity | Example | Consequence |
|---|---|---|
| **High** | `lost_or_stolen_card` predicted as `card_not_working` | The card stays active while the customer gets troubleshooting steps, so there is a risk of financial loss |
| **Medium** | `transaction_charged_twice` predicted as `request_refund` | The customer gets the wrong process and has to contact support again |
| **Low** | `card_arrival` predicted as `card_delivery_estimate` | The answer is closely related, so it is a minor inconvenience |

For this reason, the system should say when it is unsure instead of always giving an answer, and the evaluation looks at results for each intent as well as overall scores.

### 1.5 Scope

**In scope**

- English messages about the 77 BANKING77 intents
- Predicting the most likely intents, with confidence scores
- Flagging messages that are off-topic or uncertain

**Out of scope**

- Carrying out actions, such as blocking a card or sending money
- Writing full chatbot replies
- Using conversation history
- Messages containing more than one request, since each example in the dataset has one label

This is a course project trained on a public dataset. It is not a production banking system.

### 1.6 Success criteria

These targets are set by this project, and later sections report whether each one was met.

| Requirement | Measure | Target |
|---|---|---|
| Correct classification | Macro F1 on the test set | At least 0.90 |
| Reported performance | Accuracy, precision, recall and F1 | All reported on the test set |
| Better than a simple approach | Macro F1 compared with the TF-IDF baseline | Higher than the baseline |
| Consistent across intents | Lowest-scoring intents | Identified and explained |
| Fast enough for a chatbot | Prediction time for one message on a CPU | Under 200 ms |
| Handles off-topic messages | Off-topic messages flagged, and banking messages wrongly flagged | At least 80% flagged, with under 10% wrongly flagged |
| Usable by a person | Working demo | A typed message returns the top intents with confidence scores |

## 2. Setup

Select a GPU first: **Runtime > Change runtime type > T4 GPU**. Then run these cells in order.

### 2.1 Install libraries

In [ ]:
%pip install -q transformers==5.21.0 datasets==4.8.5 evaluate==0.4.6 accelerate==1.15.0 scikit-learn==1.6.1 gradio==6.30.0

### 2.2 Record library versions

These versions are pinned in `requirements.txt` so that the notebook can be reproduced.

In [ ]:
import platform
from importlib.metadata import version

PACKAGES = [
    "transformers", "datasets", "evaluate", "accelerate", "torch",
    "scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "gradio",
]

print(f"# Python {platform.python_version()}")
for package in PACKAGES:
    print(f"{package}=={version(package)}")

### 2.3 Check the GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. Select Runtime > Change runtime type > T4 GPU, then run the notebook again."
    )

DEVICE = torch.device("cuda")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### 2.4 Connect Google Drive

Trained models are saved to Drive so that they survive Colab disconnects.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/banking-intent-classifier")
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

for directory in (MODELS_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

test_file = PROJECT_DIR / "write_test.txt"
test_file.write_text("ok")
assert test_file.read_text() == "ok"
test_file.unlink()

print(f"Drive is ready: {PROJECT_DIR}")

### 2.5 Set the random seed

A fixed seed makes results repeatable.

In [ ]:
import random

import numpy as np

SEED = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f"Seed: {SEED}")

## 3. Load the data

### 3.1 About the dataset

**BANKING77** (Casanueva et al., 2020, "Efficient Intent Detection with Dual Sentence Encoders") contains online banking queries, each labelled with one of 77 intents. It is released under the CC BY 4.0 license.

The files are loaded from the authors' repository, using a fixed commit so that the notebook always gets the same data.

In [ ]:
import json
from urllib.request import urlopen

import pandas as pd

BANKING77_COMMIT = "57ec275d8078af65b7731c2a98be812d844a6d6b"
BANKING77_URL = (
    "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/"
    f"{BANKING77_COMMIT}/banking_data"
)

# Two official intent names break the lowercase naming pattern.
LABEL_FIXES = {
    "Refund_not_showing_up": "refund_not_showing_up",
    "reverted_card_payment?": "reverted_card_payment",
}


def clean_labels(labels):
    return labels.replace(LABEL_FIXES)


def load_banking77():
    train = pd.read_csv(f"{BANKING77_URL}/train.csv")
    test = pd.read_csv(f"{BANKING77_URL}/test.csv")
    with urlopen(f"{BANKING77_URL}/categories.json") as response:
        categories = json.load(response)

    for split in (train, test):
        split["category"] = clean_labels(split["category"])

    intents = sorted(LABEL_FIXES.get(name, name) for name in categories)
    return train, test, intents


train_df, test_df, INTENTS = load_banking77()
print(f"Train messages: {len(train_df):,}")
print(f"Test messages: {len(test_df):,}")
print(f"Intents: {len(INTENTS)}")

### 3.2 Map intents to numbers

Models work with numbers, so each intent gets a fixed ID. The IDs follow alphabetical order, so they are the same on every run.

In [ ]:
LABEL2ID = {intent: index for index, intent in enumerate(INTENTS)}
ID2LABEL = {index: intent for intent, index in LABEL2ID.items()}

for split in (train_df, test_df):
    split["label"] = split["category"].map(LABEL2ID)

print(f"First intent: {ID2LABEL[0]}")
print(f"Last intent: {ID2LABEL[len(INTENTS) - 1]}")

### 3.3 Check the data

These checks stop the notebook if the data is not what the rest of the project expects.

In [ ]:
EXPECTED_TRAIN_ROWS = 10_003
EXPECTED_TEST_ROWS = 3_080
EXPECTED_INTENTS = 77


def check_data(train, test, intents):
    assert len(train) == EXPECTED_TRAIN_ROWS, "Unexpected number of training rows"
    assert len(test) == EXPECTED_TEST_ROWS, "Unexpected number of test rows"
    assert len(intents) == len(set(intents)) == EXPECTED_INTENTS, "Unexpected intent list"

    for name, split in (("train", train), ("test", test)):
        assert split.notna().all().all(), f"Missing values in {name}"
        assert split["text"].str.strip().ne("").all(), f"Empty messages in {name}"
        assert set(split["category"]) == set(intents), f"Intents in {name} do not match the list"

    overlap = set(train["text"]) & set(test["text"])
    assert not overlap, f"{len(overlap)} messages appear in both train and test"


check_data(train_df, test_df, INTENTS)
print("All data checks passed")

### 3.4 Preview the data

In [ ]:
train_df.sample(5, random_state=SEED)

## 4. Explore the data

This section uses **only the training data**. The test set stays unseen until the final comparison, so that it gives a fair measure of performance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BAR_COLOR = "#2a78d6"
SURFACE_COLOR = "#fcfcfb"
TEXT_COLOR = "#0b0b0b"
MUTED_COLOR = "#52514e"
GRID_COLOR = "#e5e4e0"

plt.rcParams.update({
    "figure.facecolor": SURFACE_COLOR,
    "axes.facecolor": SURFACE_COLOR,
    "axes.edgecolor": GRID_COLOR,
    "axes.labelcolor": MUTED_COLOR,
    "axes.titlecolor": TEXT_COLOR,
    "axes.titlelocation": "left",
    "axes.titlesize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRID_COLOR,
    "text.color": TEXT_COLOR,
    "xtick.color": MUTED_COLOR,
    "ytick.color": MUTED_COLOR,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
})


def save_figure(figure, name):
    figure.savefig(RESULTS_DIR / f"{name}.png")

### 4.1 Intent balance

In [ ]:
intent_counts = train_df["category"].value_counts()

print(f"Fewest messages for an intent: {intent_counts.min()}")
print(f"Most messages for an intent: {intent_counts.max()}")
print(f"Median messages per intent: {intent_counts.median():.0f}")
print(f"Largest intent vs smallest: {intent_counts.max() / intent_counts.min():.1f} times")

In [ ]:
def plot_intent_balance(counts, top_n=10):
    figure, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
    panels = (
        (left, counts.tail(top_n), f"{top_n} intents with the fewest messages"),
        (right, counts.head(top_n).iloc[::-1], f"{top_n} intents with the most messages"),
    )
    for axis, values, title in panels:
        axis.barh(values.index, values.values, color=BAR_COLOR, height=0.65)
        axis.set_title(title)
        axis.set_xlabel("Training messages")
        axis.grid(axis="y", visible=False)
        axis.tick_params(axis="y", length=0)
    figure.suptitle(
        "Training data is uneven: the largest intent has about five times as many messages as the smallest",
        x=0.01, ha="left", fontsize=13, fontweight="bold",
    )
    figure.tight_layout()
    return figure


figure = plot_intent_balance(intent_counts)
save_figure(figure, "intent_balance")
plt.show()

**What this shows**

- The training data is uneven, from 35 messages (`contactless_not_working`) to 187 (`card_payment_fee_charged`).
- Three intents have fewer than 60 messages: `contactless_not_working`, `virtual_card_not_working` and `card_acceptance`. These may be harder to learn.
- Two of the high-risk intents are among the smallest: `lost_or_stolen_card` (82) and `compromised_card` (86). Scores for these need checking individually.
- The test set has exactly 40 messages per intent, so it does not share this imbalance.

**What this means for the project:** macro F1 is the main metric because it counts every intent equally. The validation split must be stratified, so that small intents keep their share.

### 4.2 Message length

In [ ]:
word_counts = train_df["text"].str.split().str.len()

print(f"Median length: {word_counts.median():.0f} words")
print(f"95th percentile: {word_counts.quantile(0.95):.0f} words")
print(f"99th percentile: {word_counts.quantile(0.99):.0f} words")
print(f"Shortest: {word_counts.min()} words")
print(f"Longest: {word_counts.max()} words")

In [ ]:
def plot_message_lengths(lengths):
    median = lengths.median()
    figure, axis = plt.subplots(figsize=(10, 4))
    axis.hist(lengths, bins=range(0, int(lengths.max()) + 3, 2), color=BAR_COLOR, edgecolor=SURFACE_COLOR)
    axis.axvline(median, color=TEXT_COLOR, linewidth=1.5, linestyle="--")
    axis.annotate(
        f"Median: {median:.0f} words",
        xy=(median, 1), xycoords=("data", "axes fraction"),
        xytext=(8, -14), textcoords="offset points", color=TEXT_COLOR,
    )
    axis.set_title("Most messages are short, with a small number of much longer ones", fontweight="bold")
    axis.set_xlabel("Words in message")
    axis.set_ylabel("Training messages")
    axis.grid(axis="x", visible=False)
    figure.tight_layout()
    return figure


figure = plot_message_lengths(word_counts)
save_figure(figure, "message_lengths")
plt.show()

**What this shows**

- Messages are short: half have 10 words or fewer, which confirms the limited context noted in section 1.3.
- A small group is much longer, with 42 messages over 50 words and a maximum of 79.

**What this means for the project:** the token limit is set in the next section using each model's own tokenizer. It needs to cover the long messages without wasting memory on padding.

### 4.3 Similar intents

Each intent is represented by the average TF-IDF vector of its messages. Cosine similarity between these vectors then shows which intents use the most similar wording.

In [ ]:
from itertools import combinations

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def intent_similarity(data, intents):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    vectors = vectorizer.fit_transform(data["text"])
    centroids = np.vstack([
        np.asarray(vectors[(data["category"] == intent).to_numpy()].mean(axis=0))
        for intent in intents
    ])
    similarity = cosine_similarity(centroids)
    pairs = [
        (intents[a], intents[b], similarity[a, b])
        for a, b in combinations(range(len(intents)), 2)
    ]
    table = pd.DataFrame(pairs, columns=["intent_a", "intent_b", "similarity"])
    table = table.sort_values("similarity", ascending=False, ignore_index=True)
    table["rank"] = table.index + 1
    return table


similarity_table = intent_similarity(train_df, INTENTS)
print(f"Median similarity across all {len(similarity_table):,} pairs: {similarity_table['similarity'].median():.2f}")
similarity_table.head(10).round(2)

The pairs highlighted as similar in section 1.3 can be checked the same way:

In [ ]:
SECTION_1_PAIRS = [
    ("lost_or_stolen_card", "compromised_card"),
    ("card_arrival", "card_delivery_estimate"),
    ("pending_transfer", "transfer_not_received_by_recipient"),
]


def find_pair(table, first, second):
    match = table["intent_a"].eq(min(first, second)) & table["intent_b"].eq(max(first, second))
    return table[match]


pd.concat([find_pair(similarity_table, *pair) for pair in SECTION_1_PAIRS]).round(2)

**What this shows**

- The most similar pairs share their topic and differ in one detail, such as `top_up_failed` and `top_up_reverted`.
- The three pairs from section 1.3 rank between 49th and 74th of 2,926 pairs. Their similarity is well above the median of 0.13, but they are not the closest pairs.

**What this means for the project:** word overlap alone may not separate these intents, which supports using transformers that take context into account. This is only a prediction, because similar wording does not prove that a model will confuse two intents. The error analysis tests it against real mistakes.

### 4.4 Data quality

In [ ]:
def normalize_text(text):
    words_only = text.str.lower().str.replace(r"[^\w\s]", "", regex=True)
    return words_only.str.split().str.join(" ")


def data_quality_report(data):
    normalized = normalize_text(data["text"])
    labels_per_text = data.groupby(normalized)["category"].nunique()
    return pd.Series({
        "Exact duplicates": data["text"].duplicated().sum(),
        "Duplicates after ignoring case and punctuation": normalized.duplicated().sum(),
        "Same text with different intents": (labels_per_text > 1).sum(),
        "Messages of two words or fewer": (data["text"].str.split().str.len() <= 2).sum(),
        "Messages containing digits": data["text"].str.contains(r"\d").sum(),
        "Messages containing a currency symbol": data["text"].str.contains("[£€$]").sum(),
        "Messages entirely in lowercase": (data["text"] == data["text"].str.lower()).sum(),
    }, name="Messages")


data_quality_report(train_df).to_frame()

**What this shows**

- The data is clean: no exact duplicates, and no message appears with two different intents.
- 12 messages repeat another message once case and punctuation are ignored. All have matching labels, so they cannot cause conflicting training signals.
- Very short messages such as "Lost password" exist, and they have very little context.
- Digits and currency symbols appear in some messages and can carry meaning, such as amounts.

**What this means for the project:** cleaning should stay light, because transformers benefit from natural text. The 12 near-duplicates need handling before the validation split, so that no message appears on both sides.

### 4.5 Summary

| Finding | Decision |
|---|---|
| Uneven training data | Use macro F1 as the main metric, and stratify the validation split |
| Small high-risk intents | Report `lost_or_stolen_card` and `compromised_card` individually |
| Short messages, with a few long ones | Set the token limit from the tokenizers' own counts |
| Similar wording across intents | Test the prediction against real errors in the error analysis |
| Clean text, with meaningful digits and symbols | Keep cleaning light |
| 12 near-duplicates | Remove them before the validation split |

## 5. Preprocess and split the data

This section applies the decisions from the exploration: light cleaning, removing near-duplicates, a stratified validation split, and tokenizing the text for each model.

### 5.1 Clean the text

Cleaning only tidies whitespace, because some messages contain line breaks, double spaces or non-breaking spaces. Case, punctuation, digits and currency symbols all stay, because transformers can use them.

In [ ]:
def clean_text(text):
    return text.str.split().str.join(" ")


for split in (train_df, test_df):
    changed = (clean_text(split["text"]) != split["text"]).sum()
    split["text"] = clean_text(split["text"])
    print(f"Messages with whitespace tidied: {changed}")

### 5.2 Remove near-duplicates from the training data

Two kinds of near-duplicate are removed, ignoring case and punctuation:

- **Repeats within the training data**, so that no message can end up in both the training and validation splits.
- **Copies of test messages**, because a model that has already seen a test message would get an unfairly easy question.

Only training rows are removed. The test set stays exactly as published, so the results can be compared with other work on BANKING77.

In [ ]:
def remove_near_duplicates(train, test):
    train_keys = normalize_text(train["text"])
    test_keys = set(normalize_text(test["text"]))

    repeated = train_keys.duplicated()
    in_test = train_keys.isin(test_keys)

    print(f"Repeats within the training data: {repeated.sum()}")
    print(f"Copies of test messages: {in_test.sum()}")
    return train[~(repeated | in_test)].reset_index(drop=True)


unique_train_df = remove_near_duplicates(train_df, test_df)
print(f"Training messages kept: {len(unique_train_df):,} of {len(train_df):,}")

### 5.3 Create the validation split

The validation split is used to tune the models and choose between them, so that the test set is only used once at the end. The split is **stratified**, which keeps each intent's share the same on both sides.

In [ ]:
from sklearn.model_selection import train_test_split

VALIDATION_SIZE = 0.1

train_split, val_split = train_test_split(
    unique_train_df,
    test_size=VALIDATION_SIZE,
    stratify=unique_train_df["label"],
    random_state=SEED,
)
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

split_sizes = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "messages": [len(train_split), len(val_split), len(test_df)],
    "fewest_per_intent": [
        split["label"].value_counts().min() for split in (train_split, val_split, test_df)
    ],
    "most_per_intent": [
        split["label"].value_counts().max() for split in (train_split, val_split, test_df)
    ],
})
split_sizes

These checks stop the notebook if any message appears in more than one split, or if an intent is missing.

In [ ]:
def check_splits(train, val, test, intents):
    keys = {
        "train": set(normalize_text(train["text"])),
        "validation": set(normalize_text(val["text"])),
        "test": set(normalize_text(test["text"])),
    }
    for first, second in combinations(keys, 2):
        shared = keys[first] & keys[second]
        assert not shared, f"{len(shared)} messages appear in both {first} and {second}"

    for name, split in (("train", train), ("validation", val), ("test", test)):
        assert set(split["category"]) == set(intents), f"Intents missing from {name}"


check_splits(train_split, val_split, test_df, INTENTS)
print("All split checks passed")

### 5.4 Choose the token limit

Models read tokens rather than words, and each model splits text differently, so lengths are measured with each model's own tokenizer on the training split.

In [ ]:
from transformers import AutoTokenizer

MODEL_CHECKPOINTS = {
    "DistilBERT": "distilbert-base-uncased",
    "RoBERTa": "roberta-base",
}

tokenizers = {
    name: AutoTokenizer.from_pretrained(checkpoint)
    for name, checkpoint in MODEL_CHECKPOINTS.items()
}


def token_lengths(tokenizer, texts):
    return pd.Series([len(ids) for ids in tokenizer(texts.tolist())["input_ids"]])


length_summary = pd.DataFrame({
    name: token_lengths(tokenizer, train_split["text"]).describe(percentiles=[0.5, 0.99])
    for name, tokenizer in tokenizers.items()
}).loc[["50%", "99%", "max"]].round(0).astype(int)
length_summary

The longest training message is under 100 tokens for both models, so a limit of **128** cuts off nothing in the training split. Each batch is later padded only to its own longest message, so short messages do not waste memory despite the higher limit.

In [ ]:
MAX_LENGTH = 128

for name, tokenizer in tokenizers.items():
    longest = token_lengths(tokenizer, train_split["text"]).max()
    assert longest <= MAX_LENGTH, f"{name} would truncate training messages"

print(f"Token limit: {MAX_LENGTH}")

### 5.5 Tokenize the data

Each model gets its own tokenized copy of all three splits. The TF-IDF baseline does not use these, as it works directly from the cleaned text.

In [ ]:
from datasets import Dataset, DatasetDict


def build_dataset(train, val, test):
    return DatasetDict({
        name: Dataset.from_pandas(split[["text", "label"]], preserve_index=False)
        for name, split in (("train", train), ("validation", val), ("test", test))
    })


def tokenize_dataset(dataset, tokenizer):
    def tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

    return dataset.map(tokenize_batch, batched=True, remove_columns=["text"])


dataset = build_dataset(train_split, val_split, test_df)
tokenized_datasets = {
    name: tokenize_dataset(dataset, tokenizer)
    for name, tokenizer in tokenizers.items()
}
dataset

The same message looks different to each model:

In [ ]:
EXAMPLE_MESSAGE = "Why was I charged £5 twice?"

for name, tokenizer in tokenizers.items():
    ids = tokenizer(EXAMPLE_MESSAGE)["input_ids"]
    print(f"{name}: {tokenizer.convert_ids_to_tokens(ids)}")

**What this shows**

- DistilBERT lowercases everything, so "Why" and "why" are the same token to it.
- RoBERTa keeps capital letters, and marks the start of a new word with `Ġ`.

This is one reason for comparing the two models: they do not receive quite the same information.